# Episode 15: Connect to External Tools (MCP)

Instead of writing custom tool functions for every API, what if there was a universal standard? That's MCP.

**By the end of this episode, you will:**

- Connect an AG2 agent to an MCP server and let it discover tools automatically
- Route a single agent across multiple MCP servers simultaneously
- Build your own MCP server in about 10 lines of Python
- Understand the trade-offs between stdio and SSE transports

> This episode has a companion code folder at `ep15_mcp/`.

## What Is MCP?

Think of MCP as the USB-C of AI tooling. Before USB-C, every device had its own charger and cable. MCP does the same thing for agents and tools — one protocol, universal compatibility.

The **Model Context Protocol** lets any agent discover and call tools exposed by any MCP server, without writing custom integration code for each one. The server advertises what it can do; the agent figures out when to use it.

```
┌──────────┐     MCP Protocol     ┌──────────────┐
│  AG2     │ ──────────────────> │  MCP Server   │
│  Agent   │ <────────────────── │  (any tools)  │
└──────────┘                     └──────────────┘
```

MCP adoption has been growing rapidly across the ecosystem. Major IDEs, AI assistants, and developer tools are adding MCP support, which means the number of ready-to-use tool servers keeps expanding. Writing an integration once as an MCP server makes it available to every MCP-compatible agent — not just yours.

### Two Transport Types

MCP supports two ways for agents to talk to servers, each suited to different deployment scenarios.

| Transport | How it works | Best for |
|-----------|-------------|----------|
| **stdio** | Spawns a local process, communicates via stdin/stdout | Local tools, development |
| **SSE** | Connects to a remote HTTP server via Server-Sent Events | Remote services, shared tools |

### AG2's Approach

`MCPClientSessionManager` opens sessions **on-demand** — no persistent connections sitting idle. Your agent only connects to the MCP server when it actually needs a tool, keeping resource usage low.

In [ ]:
# MCP requires Python 3.12+ and the mcp extra
!pip install "ag2[openai,mcp]" python-dotenv arxiv wikipedia -q

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## Part 1: Connect to an MCP Server

AG2's `create_toolkit` handles the entire connection lifecycle: spawn the server, discover its tools, and register them with your agent. Three lines of setup and your agent has new capabilities it didn't have before.

The example below uses **stdio transport** to launch a local MCP server process. It requires the companion files in `ep15_mcp/` to run.

> See `ep15_mcp/README.md` for setup instructions.

In [ ]:
# This requires the MCP server files in ep15_mcp/
# See ep15_mcp/README.md for setup instructions

from autogen.mcp import create_toolkit

llm_config = LLMConfig(api_type="openai", model="gpt-4o-mini")

# Connect to an MCP server (stdio transport)
toolkit = create_toolkit(server_params={
    "command": "python",
    "args": ["ep15_mcp/mcp_server.py", "stdio"]
})

with llm_config:
    agent = ConversableAgent(
        name="research_agent",
        system_message="You are a research assistant. Use available tools to find information.",
        human_input_mode="NEVER",
    )

# Register all tools from the MCP server
toolkit.register(agent)

### What Happens Under the Hood

When you call `create_toolkit`, AG2 spawns the MCP server process (or connects to a remote SSE endpoint) and asks it to list every tool it exposes — names, descriptions, parameter schemas, the works.

After `toolkit.register(agent)`, those tools appear to the agent exactly like any locally defined function. When the agent decides to invoke one, `MCPClientSessionManager` opens a session, sends the request, and returns the result. The agent never needs to know whether a tool is local Python or a remote MCP call.

## Part 2: Multi-Server Routing

A single agent can connect to **multiple MCP servers** at the same time. This is where MCP's design really shines — your agent sees a unified toolbox, even though the tools live on completely different servers.

```
                    ┌─────────────────┐
                ┌──>│  arXiv Server   │  (search_papers, get_abstract)
┌──────────┐    │   └─────────────────┘
│  Agent   │────┤
└──────────┘    │   ┌─────────────────┐
                └──>│  Wikipedia Server│  (search_wiki, get_article)
                    └─────────────────┘
```

The agent picks the right server's tools based on context. A query about transformer architectures routes to arXiv; a question about the Eiffel Tower routes to Wikipedia. No routing logic needed — the LLM reads the tool descriptions and decides.

Wiring it up is straightforward: call `create_toolkit` and `register` once per server.

```python
arxiv_toolkit = create_toolkit(server_params={"command": "python", "args": ["arxiv_server.py"]})
wiki_toolkit = create_toolkit(server_params={"command": "python", "args": ["wiki_server.py"]})

arxiv_toolkit.register(agent)
wiki_toolkit.register(agent)
```

The agent sees all tools from both servers in a single list and picks the right one automatically.

## Part 3: Build Your Own MCP Server

Building an MCP server takes about as much effort as writing a decorated Flask route. The **FastMCP** library provides a decorator pattern that turns ordinary Python functions into MCP tools.

In [ ]:
# Save this as my_mcp_server.py

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MyServer")


@mcp.tool()
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run()

That is the entire server. The `@mcp.tool()` decorator exposes the function, and the docstring becomes the tool description that agents see when they connect.

To hook it up to an agent:

```python
toolkit = create_toolkit(server_params={
    "command": "python",
    "args": ["my_mcp_server.py"]
})
toolkit.register(agent)
```

> **Python version note:** MCP requires **Python 3.12 or newer**. If you are on an earlier version, upgrading is the simplest fix. See `troubleshooting.md` for help with common setup issues.

## stdio vs SSE: Picking a Transport

Both transports speak the same protocol — the difference is where the server runs and how you connect to it.

| Aspect | stdio | SSE |
|--------|-------|-----|
| **Connection** | Local subprocess | Remote HTTP server |
| **Latency** | Very low | Network-dependent |
| **Setup** | Just a Python file | Running server needed |
| **Sharing** | Single process only | Multiple clients |
| **Best for** | Development, local tools | Production, shared services |

During development, stdio is the path of least resistance — no server to deploy, no ports to manage. When you are ready to share a tool server across multiple agents or teams, switch to SSE.

For SSE transport, the connection is just a URL:

```python
toolkit = create_toolkit(server_params={
    "url": "http://localhost:8000/sse"
})
```

## MCPClientSessionManager

Under the hood, `create_toolkit` delegates to `MCPClientSessionManager`, which handles session lifecycle: opening connections on demand, reconnecting for SSE transport, and cleaning up resources when you are done.

For most use cases, `create_toolkit` is all you need. Reach for `MCPClientSessionManager` directly when you need custom session behavior — for example, connection pooling or authentication middleware.

For a deep dive into multi-server session management, see the AG2 blog:
[AG2 MultiMCPSessionManager](https://ag2.ai/blog/2025-12-24-AG2-MultiMCPSessionManager)

## Try It Yourself

Add a second tool to the MCP server above — for example, one that returns the current date and time:

```python
@mcp.tool()
def current_time() -> str:
    """Return the current date and time."""
    from datetime import datetime
    return datetime.now().isoformat()
```

Then connect an agent to the server and ask it to greet you *and* tell you the time in a single conversation.

In [ ]:
# Your code here — extend the MCP server and test with an agent


## What's Next

In **Episode 16**, you will learn to see what your agents are doing with **observability** — tracing, logging, and debugging multi-agent workflows.